In [ ]:
#Importing all the necessary libraries
from pyspark.sql.functions import col, sha2, date_trunc, when
from pyspark.sql.types import *

# Disable vectorized reading
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")
spark.conf.set("spark.sql.legacy.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.datetimeRebaseModeInRead", "CORRECTED")

# List of all 12 months
months = [
    "2023-01", "2023-02", "2023-03", "2023-04",
    "2023-05", "2023-06", "2023-07", "2023-08",
    "2023-09", "2023-10", "2023-11", "2023-12"
]

failed_months = []
total_rows = 0

for m in months:
    try:
        print(f"Processing {m}...")
        
        # Read from curated folder
        file_path = f"{curated_path}yellow_tripdata_{m}.parquet"
        df = spark.read \
            .option("inferSchema", "false") \
            .parquet(file_path)
        
        # Cast columns
        df = df.select(
            col("VendorID").cast("long"),
            col("tpep_pickup_datetime").cast("timestamp"),
            col("tpep_dropoff_datetime").cast("timestamp"),
            col("passenger_count").cast("double"),
            col("trip_distance").cast("double"),
            col("RatecodeID").cast("double"),
            col("store_and_fwd_flag").cast("string"),
            col("PULocationID").cast("long"),
            col("DOLocationID").cast("long"),
            col("payment_type").cast("long"),
            col("fare_amount").cast("double"),
            col("extra").cast("double"),
            col("mta_tax").cast("double"),
            col("tip_amount").cast("double"),
            col("tolls_amount").cast("double"),
            col("improvement_surcharge").cast("double"),
            col("total_amount").cast("double"),
            col("congestion_surcharge").cast("double"),
            col("airport_fee").cast("double"),
            col("pickup_hour").cast("integer"),
            col("pickup_dayofweek").cast("integer"),
            col("pickup_month").cast("integer"),
            col("trip_duration_mins").cast("double"),
            col("tip_percentage").cast("double")
        )

        # ANONYMISATION TECHNIQUES
        # 1. Hash VendorID to pseudonymise the vendor
        df = df.withColumn("VendorID", 
            sha2(col("VendorID").cast("string"), 256))

        # 2. Round timestamps to nearest hour to reduce precision
        df = df.withColumn("tpep_pickup_datetime",
            date_trunc("hour", col("tpep_pickup_datetime"))) \
               .withColumn("tpep_dropoff_datetime",
            date_trunc("hour", col("tpep_dropoff_datetime")))

        # 3. Generalise location IDs into zones
        df = df.withColumn("pickup_zone",
            when(col("PULocationID") <= 100, "Zone A")
            .when(col("PULocationID") <= 200, "Zone B")
            .otherwise("Zone C")) \
               .withColumn("dropoff_zone",
            when(col("DOLocationID") <= 100, "Zone A")
            .when(col("DOLocationID") <= 200, "Zone B")
            .otherwise("Zone C")) \
               .drop("PULocationID", "DOLocationID")

        # Count rows
        row_count = df.count()
        total_rows += row_count

        #Save anonymised data
        output_path = f"{curated_path}anonymised/yellow_tripdata_{m}.parquet"
        df.write.mode("overwrite").parquet(output_path)

        print(f"✅ {m} done! Rows: {row_count:,}")

    except Exception as e:
        print(f"❌ {m} failed: {str(e)[:200]}")
        failed_months.append(m)

print(f"Total number rows saved: {total_rows:,}")
print(f"Failed months: {failed_months}")
print("Anonymisation complete!")